In [1]:
import numpy as np
import pandas as pd
import os
import pickle


In [2]:
def main(path_save, file_stem, pup):
    """
    Compute and return population activity distribution from raster data.

    This function:
    1. Loads peak analysis results from a saved pickle bundle.
    2. Extracts the raster matrix (time x ROIs), where entries indicate activity.
    3. Computes the population activity (network trace) as the sum of active ROIs per time point.
    4. Counts how often each level of activity (number of simultaneously active ROIs) occurs.
    5. Returns the distribution as a labeled pandas Series.

    Parameters
    ----------
    path_save : str
        Directory where the peak results file is stored.
    file_stem : str
        Base filename used to locate the peak results file.
    pup : str
        Label for the resulting Series (e.g., experimental subject or condition name).

    Returns
    -------
    pd.Series
        Series where:
        - Index = number of active ROIs (integer activity levels)
        - Values = counts of occurrences for each activity level
        - Name = provided `pup` label

    Notes
    -----
    - Activity level 0 (no active ROIs) is excluded from the output.
    - The resulting distribution summarizes how often different levels of 
      network synchrony occur over time.
    - Useful for comparing global activity patterns across recordings or conditions.
    """
    file_data = os.path.join(path_save, f"{file_stem}_peaks.pkl")
    with open(file_data, "rb") as f:
        bundle = pickle.load(f)
    raster = bundle["raster"]
    nettrace= np.sum(raster, axis=1)
    values, counts = np.unique(nettrace[nettrace > 0], return_counts=True)
    return pd.Series(index=values.astype(int), data=counts.astype(int), name=pup)

   

 

In [3]:
data_source = {
"exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT":["pup1_spont","pup2_spont"]
}

In [ ]:
path_dist = "Data"
# Load genotype information (indexed by pup ID)
genotype=pd.read_excel(os.path.join(path_dist, "genotypes_exp1_12.xlsx"), index_col=0)

# Initialize DataFrame to collect distributions for all recordings
dt=pd.DataFrame()
for folder_exp in data_source.keys():
    pups=data_source[folder_exp]
    for file_stem in pups:
        os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
        path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
        pup = folder_exp.split("_")[0] + "_" + file_stem.split("_")[0]
        # Compute distribution of population activity (counts per activity level)
        netcounts = main(path_save,file_stem,pup)
        dt=pd.concat([dt,netcounts], axis=1)
        print(f"Processing {file_stem} from {folder_exp}")
# Normalize distribution
dt=dt/dt.sum() # convert counts to probabilities
dt=dt.T
# Add genotype information for each sample
dt["genotype"]=genotype["genotype"]
#ompute group statistics (mean, std, n) by genotype
out=pd.DataFrame()
# Mean distributions
WT=dt[dt.genotype=="WT"].drop("genotype", axis=1).mean()
WT.name="WT"
HZ=dt[dt.genotype=="HZ"].drop("genotype", axis=1).mean()
HZ.name="HZ"
KO=dt[dt.genotype=="KO"].drop("genotype", axis=1).mean()
KO.name="KO"
out_mean=pd.concat([WT,HZ,KO], axis=1)

# Standard deviatio
WT=dt[dt.genotype=="WT"].drop("genotype", axis=1).std()
WT.name="WT"
HZ=dt[dt.genotype=="HZ"].drop("genotype", axis=1).std()
HZ.name="HZ"
KO=dt[dt.genotype=="KO"].drop("genotype", axis=1).std()
KO.name="KO"
out_std=pd.concat([WT,HZ,KO], axis=1)

# Sample counts (number of recordings per activity level)
WT=dt[dt.genotype=="WT"].drop("genotype", axis=1).count()
WT.name="WT"
HZ=dt[dt.genotype=="HZ"].drop("genotype", axis=1).count()
HZ.name="HZ"
KO=dt[dt.genotype=="KO"].drop("genotype", axis=1).count()
KO.name="KO"
out_n=pd.concat([WT,HZ,KO], axis=1)
# Save result
os.makedirs(os.path.join(path_dist,"NMF_analysis_results"), exist_ok=True)
out_mean.to_excel(os.path.join(path_dist,"NMF_analysis_results", "nmf_compounds_net_event_mean.xlsx"))
out_std.to_excel(os.path.join(path_dist,"NMF_analysis_results", "nmf_compounds_net_event_std.xlsx"))
out_n.to_excel(os.path.join(path_dist,"NMF_analysis_results", "nmf_compounds_net_event_number.xlsx"))


Processing pup1_spont from exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT
Processing pup2_spont from exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT
